In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')


In [2]:
train.columns

Index(['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment', 'label'],
      dtype='object')

In [3]:
import re

def add_term_indicator_columns(
    df,
    terms,
    prefix,
    case_insensitive=True
):
    df = df.copy()

    flags = re.IGNORECASE if case_insensitive else 0

    for term in terms:
        col_name = f"{prefix}_{term}"
        pattern = re.escape(term)
        df[col_name] = df["comment"].str.contains(
            pattern,
            regex=True,
            flags=flags,
            na=False
        )

    return df


In [4]:
# === Feature Engineering and Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion

X_full = train[['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", "disability", "race", "religion", "gender",'comment']]
X_full = X_full.copy()

y = train['label']

X_full["text_len"] = X_full["comment"].str.len()
X_full["word_count"] = X_full["comment"].str.split().str.len()

X_full["question_marks"] = X_full["comment"].str.count(r"\?")
X_full["exclamations"] = X_full["comment"].str.count(r"!")

X_full[["text_len", "word_count"]]  = X_full[["text_len", "word_count"]].fillna(0)
X_full[["question_marks", "exclamations",]] = X_full[["question_marks", "exclamations"]].fillna(False)

X_full = X_full.fillna('Null')

violent_terms = [
    "kill", "die", "death", "shoot", "fire", "killing", 
    "dead", "hang", "hit", "burn", "murder", "rid",
    "shot", "punch", "shooting", "guns", "suicide", "bomb",
    "gun", "fired", "kick", "knife", "killed", "torture",
    "bullet", "slap", "destroy", "stab"
]
X_full = add_term_indicator_columns(X_full,terms=violent_terms,prefix="v")

moral_terms = ["deserve", "should", "solution", "need to", "have to", "all of them"]
X_full = add_term_indicator_columns(X_full,terms=moral_terms,prefix="m")

group_terms = ["muslim", "jew", "jewish", "christian", "catholic", "women", "men", "female", "male",
                "black", "white", "asian", "immigrant", "refugee", "gay", "lesbian", "native", "indian"]

X_full = add_term_indicator_columns(X_full,terms=group_terms,prefix="g")

personal_attack = ["you", "your", "idiot", "stupid", "blind", "troll", "liar", "dumb", "moron"]

X_full = add_term_indicator_columns(X_full,terms=personal_attack,prefix="p")

base_num_features = ['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", "disability", "text_len", "word_count", "question_marks", "exclamations"]

term_feature_cols = [
    c for c in X_full.columns
    if c.startswith(("v_", "m_", "g_", "p_"))
]

num_features = base_num_features + term_feature_cols
text_feature = "comment"
cat_features = ["race", "religion", "gender"]


feature_encoder = ColumnTransformer(
    transformers=[
        ("text_vec", TfidfVectorizer(max_features=5000), text_feature),
        ("num_scaler", StandardScaler(), num_features),  # Fixed: wrapped in list
        ("cats", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
)

In [5]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import time

# Train/validation split - stratified for balanced classes
X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y, test_size=0.2, random_state=0, stratify=y
)
y_ovr = (y_train == 3).astype(int)
y_ovr_valid = (y_valid == 3).astype(int)


modelA = Pipeline([
        ("prep", feature_encoder),
        ("model", lgb.LGBMClassifier(
        objective="binary",
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=8,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        ))
    ])

modelA.fit(X_train,y_ovr)

print(classification_report(y_ovr_valid, modelA.predict(X_valid)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      1.00      0.99     38506
           1       0.76      0.36      0.49      1094

    accuracy                           0.98     39600
   macro avg       0.87      0.68      0.74     39600
weighted avg       0.98      0.98      0.98     39600



In [6]:
import numpy as np

def predict_with_threshold(model, X, class_idx=3, threshold=0.2):
    probs = model.predict_proba(X)
    y_pred = np.argmax(probs, axis=1)
    y_pred[probs[:, class_idx] > threshold] = class_idx
    return y_pred


In [7]:
y_pred = predict_with_threshold(modelA, X_valid, class_idx=1, threshold=0.2)

print(classification_report(y_ovr_valid, y_pred))
print("Macro F1:", f1_score(y_ovr_valid, y_pred, average="macro"))
#LGBM SCORE

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.99      0.99      0.99     38506
           1       0.59      0.63      0.61      1094

    accuracy                           0.98     39600
   macro avg       0.79      0.81      0.80     39600
weighted avg       0.98      0.98      0.98     39600

Macro F1: 0.7981158953281715


In [8]:
mask_non_violent = y_train != 3

X_train_B = X_train[mask_non_violent]
y_train_B = y_train[mask_non_violent]

mask_non_violent_val = y_valid != 3

X_valid_B = X_valid[mask_non_violent_val]
y_valid_B = y_valid[mask_non_violent_val]

In [9]:
modelB = Pipeline([
    ("prep", feature_encoder),
    ("model", lgb.LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=8,
        class_weight="balanced",
        n_jobs=-1,
        random_state=1,
        verbosity=-1
    ))
])

modelB.fit(X_train_B, y_train_B)

print(classification_report(y_valid_B, modelB.predict(X_valid_B)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.71      0.85      0.77      3183
           2       0.88      0.90      0.89     12488

    accuracy                           0.92     38506
   macro avg       0.86      0.90      0.87     38506
weighted avg       0.92      0.92      0.92     38506



In [16]:
def fuzzy_hierarchical_predict_proba(modelA, modelB, X, t_v=0.20):

    p_v = modelA.predict_proba(X)[:, 1]
    p012 = modelB.predict_proba(X)

    # re-scale violence probability
    p_v_adj = np.clip((p_v - t_v) / (1.0 - t_v), 0.0, 1.0)

    P = np.zeros((X.shape[0], 4))

    P[:, 3] = p_v_adj
    P[:, 0] = (1 - p_v_adj) * p012[:, 0]
    P[:, 1] = (1 - p_v_adj) * p012[:, 1]
    P[:, 2] = (1 - p_v_adj) * p012[:, 2]

    return P


In [26]:
for t in [0.05, 0.10, 0.15, 0.20, 0.25]:
    P = fuzzy_hierarchical_predict_proba(modelA, modelB, X_valid, t_v=t)
    pred = np.argmax(P, axis=1)
    f1 = f1_score(y_valid, pred, average="macro")
    print(t, f1)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.05 0.6917335744744053


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.1 0.6849459074462914


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.15 0.679146801810381


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.2 0.6723937654752674


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.25 0.6685331856872024


In [24]:
P = fuzzy_hierarchical_predict_proba(modelA, modelB, X_valid, t_v=0.1)
y_pred = np.argmax(P, axis=1)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
from sklearn.metrics import classification_report, f1_score

print(classification_report(y_valid, y_pred))
print("Macro F1:", f1_score(y_valid, y_pred, average="macro"))


              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.69      0.84      0.76      3183
           2       0.82      0.90      0.86     12488
           3       0.60      0.09      0.16      1094

    accuracy                           0.90     39600
   macro avg       0.77      0.69      0.68     39600
weighted avg       0.90      0.90      0.89     39600

Macro F1: 0.6849459074462914


In [15]:
P

array([[5.13410756e-03, 3.65187871e-02, 9.39163511e-01, 1.91835941e-02],
       [9.82895779e-01, 1.57670923e-03, 1.50736433e-02, 4.53868586e-04],
       [1.17507597e-02, 5.88537572e-01, 3.89806068e-01, 9.90560029e-03],
       ...,
       [2.91624493e-02, 5.45156527e-01, 3.52854614e-01, 7.28264097e-02],
       [5.67978980e-02, 5.00129471e-02, 8.78945805e-01, 1.42433495e-02],
       [4.88237132e-03, 6.81391523e-03, 9.71043220e-01, 1.72604938e-02]])

In [27]:
def hierarchical_hard_predict(
    modelA,
    modelB,
    X,
    t_v=0.20
):
    # violent probability
    p_v = modelA.predict_proba(X)[:, 1]

    # predictions for 0/1/2 model
    p012 = modelB.predict_proba(X)

    y_pred = np.zeros(len(X), dtype=int)

    # 1) violent gate
    is_v = p_v > t_v
    y_pred[is_v] = 3

    # 2) non-violent → use modelB
    rest = ~is_v
    y_pred[rest] = np.argmax(p012[rest], axis=1)

    return y_pred


In [29]:
y_pred = hierarchical_hard_predict(
    modelA,
    modelB,
    X_valid,
    t_v=0.1
)

print(classification_report(y_valid, y_pred))
print("macro:", f1_score(y_valid, y_pred, average="macro"))


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.94      0.96     22835
           1       0.70      0.81      0.75      3183
           2       0.85      0.83      0.84     12488
           3       0.32      0.51      0.40      1094

    accuracy                           0.88     39600
   macro avg       0.71      0.78      0.74     39600
weighted avg       0.90      0.88      0.89     39600

macro: 0.7375173895056313
